In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 4 — Pipeline Orchestrator
# MAGIC Runs Bronze → Silver → Gold in sequence.
# MAGIC
# MAGIC | Mode | When to use |
# MAGIC |------|-------------|
# MAGIC | `full_refresh` | First run, or full rebuild |
# MAGIC | `incremental`  | Weekly run — only new Year+WeekNumber rows |
# MAGIC
# MAGIC ### Databricks Job setup
# MAGIC ```
# MAGIC Workflows → Jobs → Create Job
# MAGIC   Task       : Notebook  →  /path/to/04_orchestrator
# MAGIC   Schedule   : 0 6 * * 1   (every Monday 6 AM)
# MAGIC   Parameters : {"mode": "incremental"}
# MAGIC ```

# COMMAND ----------
# MAGIC %md ## 0. Configuration + Mode

# COMMAND ----------

dbutils.widgets.text("mode", "full_refresh")
MODE = dbutils.widgets.get("mode").strip().lower()

if MODE not in ["full_refresh", "incremental"]:
    raise ValueError("mode must be 'full_refresh' or 'incremental'")

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime

current_user = spark.sql("SELECT current_user()").collect()[0][0]

CATALOG = "workspace"
SCHEMA  = "promotion"
# RAW_PATH = f"/Workspace/Users/{current_user}/bda_course/Promotion_raw_data"
RAW_PATH =f"/Volumes/workspace/default/course_data/Promotion_raw_data"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")
print(f"Raw path: {RAW_PATH}")
print(f"Mode    : {MODE}")
print(f"Run ID  : {RUN_ID}")


# COMMAND ----------
# MAGIC %md ## 1. Watermark + Run Log

# COMMAND ----------

# FIXED: Dropping tables to clear the schema mismatch causing the UNRESOLVED_COLUMN error
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.pipeline_watermarks")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.pipeline_run_log")

spark.sql(f"""
CREATE TABLE {CATALOG}.{SCHEMA}.pipeline_watermarks (
    table_name      STRING,
    last_year       INT,
    last_week       INT,
    rows_processed  BIGINT,
    run_status      STRING,
    run_mode        STRING,
    updated_at      TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE {CATALOG}.{SCHEMA}.pipeline_run_log (
    run_id       STRING,
    run_mode     STRING,
    stage        STRING,
    rows_out     BIGINT,
    status       STRING,
    error_msg    STRING,
    started_at   TIMESTAMP,
    finished_at  TIMESTAMP
)
USING DELTA
""")

def get_watermark():
    rows = spark.sql(f"""
        SELECT last_year, last_week
        FROM {CATALOG}.{SCHEMA}.pipeline_watermarks
        WHERE table_name = 'bronze_sales'
          AND run_status = 'SUCCESS'
        ORDER BY updated_at DESC
        LIMIT 1
    """).collect()
    return (int(rows[0]["last_year"]), int(rows[0]["last_week"])) if rows else (1900, 0)

def set_watermark(year, week, n, status="SUCCESS"):
    # FIXED: Added explicit column names to match table schema
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.pipeline_watermarks 
        (table_name, last_year, last_week, rows_processed, run_status, run_mode, updated_at)
        VALUES (
            'bronze_sales',
            {year},
            {week},
            {n},
            '{status}',
            '{MODE}',
            CURRENT_TIMESTAMP()
        )
    """)
    print(f"  Watermark → Year={year}, Week={week} ({n:,} rows, {status})")

def log(stage, rows=0, status="SUCCESS", error=""):
    # FIXED: Added explicit column names to match table schema
    safe_error = (error or "")[:200].replace("'", "")
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.pipeline_run_log 
        (run_id, run_mode, stage, rows_out, status, error_msg, started_at, finished_at)
        VALUES (
            '{RUN_ID}',
            '{MODE}',
            '{stage}',
            {rows},
            '{status}',
            '{safe_error}',
            CURRENT_TIMESTAMP(),
            CURRENT_TIMESTAMP()
        )
    """)


# COMMAND ----------
# MAGIC %md ## 2. Shared helpers

# COMMAND ----------

def read_raw(filename):
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(f"{RAW_PATH}/{filename}")
    )

def clean_discount(col_name):
    return (
        F.when(
            F.trim(F.col(col_name)).endswith("%"),
            F.regexp_extract(F.trim(F.col(col_name)), r"([\d\.]+)", 1).cast(DoubleType()) / 100
        ).when(
            F.col(col_name).cast(DoubleType()) > 1,
            F.col(col_name).cast(DoubleType()) / 100
        ).otherwise(
            F.col(col_name).cast(DoubleType())
        )
    )

def write_table(df, full_name, mode="overwrite"):
    (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )

def table_exists(full_name: str) -> bool:
    try:
        spark.table(full_name)
        return True
    except Exception:
        return False


# COMMAND ----------
# MAGIC %md ## 3. Bronze Stage

# COMMAND ----------

def run_bronze():
    print("\n▓▓▓  BRONZE  ▓▓▓")
    t = datetime.now()

    def ingest(filename, table, mode="overwrite"):
        full_name = f"{CATALOG}.{SCHEMA}.bronze_{table}"

        df = read_raw(filename)
        df = (
            df.withColumn("_ingest_time", F.current_timestamp())
              .withColumn("_source_file", F.col("_metadata.file_path"))
              .withColumn("_batch_date", F.current_date())
        )

        write_table(df, full_name, mode=mode)

        n = spark.table(full_name).count()
        print(f"  {full_name:45s} {n:>8,} rows")
        return n

    # Dimensions: always full overwrite
    ingest("RAW_Product.csv",   "product", mode="overwrite")
    ingest("RAW_Store.csv",     "store", mode="overwrite")
    ingest("RAW_Date.csv",      "date", mode="overwrite")
    ingest("RAW_Promotion.csv", "promotion", mode="overwrite")

    # Fact: incremental on Year + WeekNumber
    full_name = f"{CATALOG}.{SCHEMA}.bronze_sales"

    df_raw = (
        read_raw("RAW_Sales.csv")
        .withColumn("_y", F.col("Year").cast(IntegerType()))
        .withColumn("_w", F.col("WeekNumber").cast(IntegerType()))
    )

    if MODE == "incremental":
        wm_year, wm_week = get_watermark()
        print(f"  Watermark: Year={wm_year}, Week={wm_week}")

        df_new = df_raw.filter(
            (F.col("_y") > wm_year) |
            ((F.col("_y") == wm_year) & (F.col("_w") > wm_week))
        )
        n = df_new.count()

        if n == 0:
            print("  bronze_sales                                  0 new rows — skipping")
            log("bronze", rows=0)
            return

        write_mode = "append"
    else:
        df_new = df_raw
        n = df_new.count()
        write_mode = "overwrite"

    df_out = (
        df_new.drop("_y", "_w")
              .withColumn("_ingest_time", F.current_timestamp())
              .withColumn("_source_file", F.col("_metadata.file_path"))
              .withColumn("_batch_date", F.current_date())
    )

    write_table(df_out, full_name, mode=write_mode)

    max_row = df_new.orderBy(F.desc("_y"), F.desc("_w")).first()
    set_watermark(int(max_row["_y"]), int(max_row["_w"]), n)

    print(f"  {full_name:45s} {n:>8,} rows")

    log("bronze", rows=n)
    print(f"  ✔ {(datetime.now() - t).seconds}s")


# COMMAND ----------
# MAGIC %md ## 4. Silver Stage

# COMMAND ----------

def run_silver():
    print("\n▓▓▓  SILVER  ▓▓▓")
    t = datetime.now()

    BRONZE_PRODUCT    = f"{CATALOG}.{SCHEMA}.bronze_product"
    BRONZE_STORE      = f"{CATALOG}.{SCHEMA}.bronze_store"
    BRONZE_DATE       = f"{CATALOG}.{SCHEMA}.bronze_date"
    BRONZE_PROMOTION  = f"{CATALOG}.{SCHEMA}.bronze_promotion"
    BRONZE_SALES      = f"{CATALOG}.{SCHEMA}.bronze_sales"

    SILVER_DIM_PRODUCT    = f"{CATALOG}.{SCHEMA}.silver_dim_product"
    SILVER_DIM_STORE      = f"{CATALOG}.{SCHEMA}.silver_dim_store"
    SILVER_DIM_DATE       = f"{CATALOG}.{SCHEMA}.silver_dim_date"
    SILVER_DIM_PROMOTION  = f"{CATALOG}.{SCHEMA}.silver_dim_promotion"
    SILVER_FACT_SALES     = f"{CATALOG}.{SCHEMA}.silver_fact_sales"

    # dim_product
    df = spark.read.table(BRONZE_PRODUCT).drop("_ingest_time", "_source_file", "_batch_date")
    df = (
        df.withColumn("Product",  F.initcap(F.trim(F.col("Product"))))
          .withColumn("Brand",    F.initcap(F.trim(F.col("Brand"))))
          .withColumn("Category", F.initcap(F.trim(F.col("Category"))))
          .withColumn("Size",     F.trim(F.col("Size")))
          .withColumn("Supplier", F.initcap(F.trim(F.col("Supplier"))))
          .withColumn("UnitCost", F.col("UnitCost").cast(DoubleType()))
          .dropDuplicates(["Product"])
          .withColumn("ProductKey", F.row_number().over(Window.orderBy("Product")))
          .select("ProductKey", "Product", "Brand", "Category", "Size", "Supplier", "UnitCost")
    )
    write_table(df, SILVER_DIM_PRODUCT)
    print(f"  {SILVER_DIM_PRODUCT:45s} {spark.table(SILVER_DIM_PRODUCT).count():>8,} rows")

    # dim_store
    df = spark.read.table(BRONZE_STORE).drop("_ingest_time", "_source_file", "_batch_date")
    df = (
        df.withColumn("StoreID",        F.trim(F.col("StoreID")))
          .withColumn("StoreName",      F.initcap(F.trim(F.col("StoreName"))))
          .withColumn("City",           F.initcap(F.trim(F.col("City"))))
          .withColumn("Province",       F.initcap(F.trim(F.col("Province"))))
          .withColumn("ProvinceAbbrev", F.upper(F.trim(F.col("ProvinceAbbrev"))))
          .withColumn("Country",        F.initcap(F.trim(F.col("Country"))))
          .dropDuplicates(["StoreID"])
          .withColumn("StoreKey", F.row_number().over(Window.orderBy("StoreID")))
          .select("StoreKey", "StoreID", "StoreName", "City", "Province", "ProvinceAbbrev", "Country")
    )
    write_table(df, SILVER_DIM_STORE)
    print(f"  {SILVER_DIM_STORE:45s} {spark.table(SILVER_DIM_STORE).count():>8,} rows")

    # dim_date
    df = spark.read.table(BRONZE_DATE).drop("_ingest_time", "_source_file", "_batch_date")
    df = (
        df.withColumn("Year",        F.col("Year").cast(IntegerType()))
          .withColumn("WeekNumber",  F.col("WeekNumber").cast(IntegerType()))
          .withColumn("MonthNumber", F.col("MonthNumber").cast(IntegerType()))
          .withColumn(
              "FiscalYear",
              F.concat(F.lit("FY"), F.regexp_extract(F.col("FiscalYear"), r"(\d{4})", 1))
          )
          .withColumn("Month", F.initcap(F.trim(F.col("Month"))))
          .withColumn(
              "Quarter",
              F.when(F.col("Quarter").isNotNull(), F.col("Quarter"))
               .when(F.col("MonthNumber").isin([2, 3, 4]),  F.lit("Q1"))
               .when(F.col("MonthNumber").isin([5, 6, 7]),  F.lit("Q2"))
               .when(F.col("MonthNumber").isin([8, 9, 10]), F.lit("Q3"))
               .otherwise(F.lit("Q4"))
          )
          .dropDuplicates(["Year", "WeekNumber"])
          .withColumn("DateKey", (F.col("Year") * 100 + F.col("WeekNumber")).cast(IntegerType()))
          .select("DateKey", "Year", "WeekNumber", "WeekStartDate", "Month", "MonthNumber", "Quarter", "FiscalYear")
    )
    write_table(df, SILVER_DIM_DATE)
    print(f"  {SILVER_DIM_DATE:45s} {spark.table(SILVER_DIM_DATE).count():>8,} rows")

    # dim_promotion
    df = spark.read.table(BRONZE_PROMOTION).drop("_ingest_time", "_source_file", "_batch_date")
    df = (
        df.withColumn("Discount",      clean_discount("Discount"))
          .withColumn("PromotionName", F.trim(F.col("PromotionName")))
          .withColumn("OnFlyer",       F.initcap(F.trim(F.col("OnFlyer"))))
          .withColumn("PromotionType", F.initcap(F.trim(F.col("PromotionType"))))
          .withColumn("DiscountTier",  F.initcap(F.trim(F.col("DiscountTier"))))
          .withColumn(
              "DiscountTier",
              F.when(F.col("DiscountTier").isNotNull(), F.col("DiscountTier"))
               .when(F.col("Discount") >= 0.30, F.lit("Deep"))
               .when(F.col("Discount") > 0.00,  F.lit("Mid"))
               .otherwise(F.lit("None"))
          )
          .dropDuplicates(["PromotionName"])
          .withColumn("PromotionKey", F.row_number().over(Window.orderBy("PromotionName")))
          .select("PromotionKey", "PromotionName", "OnFlyer", "Discount", "PromotionType", "DiscountTier")
    )
    write_table(df, SILVER_DIM_PROMOTION)
    print(f"  {SILVER_DIM_PROMOTION:45s} {spark.table(SILVER_DIM_PROMOTION).count():>8,} rows")

    # fact_sales
    df_b = spark.read.table(BRONZE_SALES).drop("_ingest_time", "_source_file", "_batch_date")

    dim_prod  = spark.read.table(SILVER_DIM_PRODUCT).select("ProductKey", "Product", "UnitCost")
    dim_store = spark.read.table(SILVER_DIM_STORE).select("StoreKey", "StoreID")
    dim_date  = spark.read.table(SILVER_DIM_DATE).select("DateKey", "Year", "WeekNumber")
    dim_promo = spark.read.table(SILVER_DIM_PROMOTION).select("PromotionKey", "PromotionName")

    df_b = (
        df_b.withColumn("Product",      F.initcap(F.trim(F.col("Product"))))
            .withColumn("OnFlyer",      F.initcap(F.trim(F.col("OnFlyer"))))
            .withColumn("Discount",     clean_discount("Discount"))
            .withColumn("Year",         F.col("Year").cast(IntegerType()))
            .withColumn("WeekNumber",   F.col("WeekNumber").cast(IntegerType()))
            .withColumn("StoreID",      F.trim(F.col("StoreID")))
            .withColumn("Price",        F.col("Price").cast(DoubleType()))
            .withColumn("Units",        F.col("Units").cast(IntegerType()))
            .withColumn("SalesAmt",     F.col("SalesAmt").cast(DoubleType()))
            .withColumn("GrossMargin",  F.col("GrossMargin").cast(DoubleType()))
            .withColumn("Transactions", F.col("Transactions").cast(IntegerType()))
            .dropDuplicates(["Year", "WeekNumber", "StoreID", "Product"])
            .withColumn(
                "PromotionName",
                F.when(F.col("Discount") == 0, F.lit("No Promotion"))
                 .when(
                     F.col("OnFlyer") == "Yes",
                     F.concat(
                         (F.col("Discount") * 100).cast(IntegerType()).cast(StringType()),
                         F.lit("% Off + Flyer")
                     )
                 )
                 .otherwise(
                     F.concat(
                         (F.col("Discount") * 100).cast(IntegerType()).cast(StringType()),
                         F.lit("% Off")
                     )
                 )
            )
    )

    df_fact = (
        df_b.join(dim_prod,  on="Product",              how="left")
            .join(dim_store, on="StoreID",              how="left")
            .join(dim_date,  on=["Year", "WeekNumber"], how="left")
            .join(dim_promo, on="PromotionName",        how="left")
            .withColumn(
                "Price",
                F.when(F.col("Price").isNull(), F.round(F.col("SalesAmt") / F.col("Units"), 2))
                 .otherwise(F.col("Price"))
            )
            .withColumn("IsBelowCost", F.when(F.col("GrossMargin") < 0, 1).otherwise(0))
            .select(
                "ProductKey",
                "StoreKey",
                "DateKey",
                "PromotionKey",
                "Year",
                "WeekNumber",
                F.col("Price").alias("ActualPrice"),
                F.col("Discount").alias("DiscountPct"),
                F.col("Units").alias("UnitsSold"),
                "SalesAmt",
                "GrossMargin",
                "Transactions",
                "IsBelowCost"
            )
    )

    if MODE == "incremental" and table_exists(SILVER_FACT_SALES):
        tbl = DeltaTable.forName(spark, SILVER_FACT_SALES)
        (
            tbl.alias("e")
               .merge(
                   df_fact.alias("n"),
                   """
                   e.ProductKey = n.ProductKey AND
                   e.StoreKey = n.StoreKey AND
                   e.DateKey = n.DateKey AND
                   e.PromotionKey = n.PromotionKey
                   """
               )
               .whenMatchedUpdateAll()
               .whenNotMatchedInsertAll()
               .execute()
        )
    else:
        write_table(df_fact, SILVER_FACT_SALES, mode="overwrite")

    n = spark.table(SILVER_FACT_SALES).count()
    print(f"  {SILVER_FACT_SALES:45s} {n:>8,} rows")

    log("silver", rows=n)
    print(f"  ✔ {(datetime.now() - t).seconds}s")


# COMMAND ----------
# MAGIC %md ## 5. Gold Stage

# COMMAND ----------

def run_gold():
    print("\n▓▓▓  GOLD  ▓▓▓")
    t = datetime.now()

    fact    = spark.read.table(f"{CATALOG}.{SCHEMA}.silver_fact_sales")
    d_prod  = spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_product")
    d_store = spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_store")
    d_date  = spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_date")
    d_promo = spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_promotion")

    def wg(df, name):
        full_name = f"{CATALOG}.{SCHEMA}.gold_{name}"
        write_table(df, full_name)
        print(f"  {full_name:45s} {spark.table(full_name).count():>8,} rows")

    # gold_price_elasticity
    df = (
        fact.join(d_prod.select("ProductKey", "Product", "UnitCost"), "ProductKey")
            .join(d_promo.select("PromotionKey", "PromotionName", "OnFlyer", "DiscountTier"), "PromotionKey")
            .groupBy(
                "ProductKey", "Product", "UnitCost",
                "PromotionKey", "PromotionName", "OnFlyer", "DiscountTier",
                "ActualPrice", "DiscountPct", "Year", "WeekNumber"
            )
            .agg(
                F.sum("UnitsSold").alias("ChainUnits"),
                F.sum("SalesAmt").alias("ChainSalesAmt"),
                F.sum("GrossMargin").alias("ChainGrossMargin"),
                F.sum("Transactions").alias("ChainTransactions"),
                F.sum("IsBelowCost").alias("StoresBelowCost"),
                F.count("StoreKey").alias("StoresActive")
            )
            .groupBy(
                "ProductKey", "Product", "UnitCost",
                "PromotionName", "OnFlyer", "DiscountTier",
                "ActualPrice", "DiscountPct"
            )
            .agg(
                F.count("WeekNumber").alias("WeeksAtThisPrice"),
                F.round(F.avg("ChainUnits"), 0).alias("AvgWeeklyUnits"),
                F.round(F.sum("ChainUnits"), 0).alias("TotalUnits"),
                F.round(F.avg("ChainSalesAmt"), 2).alias("AvgWeeklySales"),
                F.round(F.sum("ChainSalesAmt"), 2).alias("TotalSales"),
                F.round(F.avg("ChainGrossMargin"), 2).alias("AvgWeeklyMargin"),
                F.round(F.sum("ChainGrossMargin"), 2).alias("TotalMargin"),
                F.round(F.avg("ChainTransactions"), 0).alias("AvgWeeklyTransactions"),
                F.round(F.avg("StoresActive"), 0).alias("AvgStoresActive")
            )
            .withColumn("GrossMarginPct", F.round(F.col("TotalMargin") / F.col("TotalSales") * 100, 1))
            .withColumn("MarginPerUnit", F.round(F.col("AvgWeeklyMargin") / F.col("AvgWeeklyUnits"), 4))
            .withColumn("RankByUnits", F.rank().over(Window.partitionBy("Product").orderBy(F.desc("AvgWeeklyUnits"))))
            .withColumn("RankByMargin", F.rank().over(Window.partitionBy("Product").orderBy(F.desc("AvgWeeklyMargin"))))
            .withColumn("_gold_ts", F.current_timestamp())
            .orderBy("Product", "ActualPrice")
    )
    wg(df, "price_elasticity")

    # gold_promotion_uplift
    baseline = (
        fact.join(d_promo.select("PromotionKey", "Discount"), "PromotionKey")
            .filter(F.col("Discount") == 0)
            .join(d_date.select("DateKey", F.col("Year").alias("DateYear"), F.col("WeekNumber").alias("DateWeekNumber")), "DateKey")
            .groupBy("ProductKey", "DateYear", "DateWeekNumber")
            .agg(
                F.sum("UnitsSold").alias("WeekUnits"),
                F.sum("SalesAmt").alias("WeekSales"),
                F.sum("GrossMargin").alias("WeekMargin")
            )
            .groupBy("ProductKey")
            .agg(
                F.round(F.avg("WeekUnits"), 0).alias("BaselineUnits"),
                F.round(F.avg("WeekSales"), 2).alias("BaselineSales"),
                F.round(F.avg("WeekMargin"), 2).alias("BaselineMargin")
            )
    )

    df = (
        fact.join(d_prod.select("ProductKey", "Product"), "ProductKey")
            .join(d_promo.select("PromotionKey", "PromotionName", "OnFlyer", "Discount", "DiscountTier"), "PromotionKey")
            .join(d_date.select("DateKey", F.col("Year").alias("DateYear"), F.col("WeekNumber").alias("DateWeekNumber")), "DateKey")
            .groupBy(
                "ProductKey", "Product", "PromotionKey", "PromotionName",
                "OnFlyer", "Discount", "DiscountTier", "DateYear", "DateWeekNumber"
            )
            .agg(
                F.sum("UnitsSold").alias("WeekUnits"),
                F.sum("SalesAmt").alias("WeekSales"),
                F.sum("GrossMargin").alias("WeekMargin"),
                F.sum("IsBelowCost").alias("StoresBelowCost")
            )
            .groupBy("ProductKey", "Product", "PromotionKey", "PromotionName", "OnFlyer", "Discount", "DiscountTier")
            .agg(
                F.count("DateWeekNumber").alias("WeeksRan"),
                F.round(F.avg("WeekUnits"), 0).alias("AvgWeeklyUnits"),
                F.round(F.avg("WeekSales"), 2).alias("AvgWeeklySales"),
                F.round(F.avg("WeekMargin"), 2).alias("AvgWeeklyMargin"),
                F.round(F.sum("WeekMargin"), 2).alias("TotalMargin"),
                F.sum("StoresBelowCost").alias("BelowCostInstances")
            )
            .join(baseline, "ProductKey", "left")
            .withColumn("UnitUpliftPct", F.round((F.col("AvgWeeklyUnits") - F.col("BaselineUnits")) / F.col("BaselineUnits") * 100, 1))
            .withColumn("SalesUpliftPct", F.round((F.col("AvgWeeklySales") - F.col("BaselineSales")) / F.col("BaselineSales") * 100, 1))
            .withColumn("MarginUpliftPct", F.round((F.col("AvgWeeklyMargin") - F.col("BaselineMargin")) / F.col("BaselineMargin") * 100, 1))
            .withColumn("IncrementalUnits", F.round(F.col("AvgWeeklyUnits") - F.col("BaselineUnits"), 0))
            .withColumn("IncrementalMargin", F.round(F.col("AvgWeeklyMargin") - F.col("BaselineMargin"), 2))
            .withColumn("_gold_ts", F.current_timestamp())
            .orderBy("Product", "Discount")
    )
    wg(df, "promotion_uplift")

    # gold_weekly_trend
    df = (
        fact.join(d_prod.select("ProductKey", "Product"), "ProductKey")
            .join(
                d_date.select(
                    "DateKey",
                    F.col("Year").alias("DateYear"),
                    F.col("WeekNumber").alias("DateWeekNumber"),
                    "Month", "MonthNumber", "Quarter", "FiscalYear", "WeekStartDate"
                ),
                "DateKey"
            )
            .join(d_promo.select("PromotionKey", "PromotionName", "OnFlyer", "Discount"), "PromotionKey")
            .groupBy(
                "Product", "DateYear", "DateWeekNumber", "Month", "MonthNumber",
                "Quarter", "FiscalYear", "WeekStartDate", "PromotionName", "OnFlyer", "Discount"
            )
            .agg(
                F.sum("UnitsSold").alias("ChainUnits"),
                F.round(F.sum("SalesAmt"), 2).alias("ChainSalesAmt"),
                F.round(F.sum("GrossMargin"), 2).alias("ChainGrossMargin"),
                F.sum("Transactions").alias("ChainTransactions"),
                F.sum("IsBelowCost").alias("StoresBelowCost"),
                F.count("StoreKey").alias("StoreCount")
            )
            .withColumn("SortDateKey", (F.col("DateYear") * 100 + F.col("DateWeekNumber")).cast("int"))
            .withColumn(
                "RollingAvg4WkUnits",
                F.round(
                    F.avg("ChainUnits").over(
                        Window.partitionBy("Product").orderBy("SortDateKey").rowsBetween(-3, 0)
                    ),
                    0
                )
            )
            .withColumn(
                "PrevWeekUnits",
                F.lag("ChainUnits", 1).over(Window.partitionBy("Product").orderBy("SortDateKey"))
            )
            .withColumn(
                "WoWChangePct",
                F.round((F.col("ChainUnits") - F.col("PrevWeekUnits")) / F.col("PrevWeekUnits") * 100, 1)
            )
            .drop("PrevWeekUnits")
            .withColumn("GrossMarginPct", F.round(F.col("ChainGrossMargin") / F.col("ChainSalesAmt") * 100, 1))
            .withColumn("IsPromoWeek", F.when(F.col("Discount") > 0, 1).otherwise(0))
            .withColumn("_gold_ts", F.current_timestamp())
            .orderBy("Product", "DateYear", "DateWeekNumber")
    )
    wg(df, "weekly_trend")

    # gold_province_summary
    df = (
        fact.join(d_prod.select("ProductKey", "Product"), "ProductKey")
            .join(d_store.select("StoreKey", "Province", "ProvinceAbbrev"), "StoreKey")
            .join(
                d_date.select(
                    "DateKey",
                    F.col("Year").alias("DateYear"),
                    F.col("WeekNumber").alias("DateWeekNumber"),
                    "FiscalYear", "Quarter"
                ),
                "DateKey"
            )
            .join(d_promo.select("PromotionKey", "PromotionName", "OnFlyer", "Discount", "DiscountTier"), "PromotionKey")
            .groupBy(
                "Product", "Province", "ProvinceAbbrev", "DateYear", "DateWeekNumber",
                "FiscalYear", "Quarter", "PromotionName", "OnFlyer", "Discount", "DiscountTier"
            )
            .agg(
                F.sum("UnitsSold").alias("TotalUnits"),
                F.round(F.sum("SalesAmt"), 2).alias("TotalSales"),
                F.round(F.sum("GrossMargin"), 2).alias("TotalMargin"),
                F.sum("Transactions").alias("TotalTransactions"),
                F.count("StoreKey").alias("StoreCount"),
                F.sum("IsBelowCost").alias("BelowCostInstances")
            )
            .withColumn("GrossMarginPct", F.round(F.col("TotalMargin") / F.col("TotalSales") * 100, 1))
            .withColumn("UnitsPerStore", F.round(F.col("TotalUnits") / F.col("StoreCount"), 1))
            .withColumn("_gold_ts", F.current_timestamp())
            .orderBy("Product", "Province", "DateYear", "DateWeekNumber", "Quarter")
    )
    wg(df, "province_summary")

    # gold_loss_leader
    aussie_keys = [
        r["ProductKey"]
        for r in spark.read.table(f"{CATALOG}.{SCHEMA}.silver_dim_product")
             .filter(F.col("Product") == "Aussie")
             .select("ProductKey")
             .collect()
    ]

    df = (
        fact.filter(F.col("ProductKey").isin(aussie_keys))
            .join(d_prod.select("ProductKey", "Product", "UnitCost"), "ProductKey")
            .join(d_promo.select("PromotionKey", "PromotionName", "OnFlyer", "Discount", "DiscountTier"), "PromotionKey")
            .join(
                d_date.select("DateKey", F.col("Year").alias("DateYear"), F.col("WeekNumber").alias("DateWeekNumber")),
                "DateKey"
            )
            .groupBy(
                "ProductKey", "Product", "UnitCost", "PromotionKey", "PromotionName",
                "OnFlyer", "Discount", "DiscountTier", "DateYear", "DateWeekNumber", "ActualPrice"
            )
            .agg(
                F.sum("UnitsSold").alias("ChainUnits"),
                F.round(F.sum("SalesAmt"), 2).alias("ChainSales"),
                F.round(F.sum("GrossMargin"), 2).alias("ChainMargin"),
                F.sum("Transactions").alias("ChainTransactions"),
                F.count("StoreKey").alias("StoreCount")
            )
            .groupBy("Product", "UnitCost", "PromotionName", "OnFlyer", "Discount", "DiscountTier", "ActualPrice")
            .agg(
                F.count("DateWeekNumber").alias("WeeksObserved"),
                F.round(F.avg("ChainUnits"), 0).alias("AvgWeeklyUnits"),
                F.round(F.avg("ChainSales"), 2).alias("AvgWeeklySales"),
                F.round(F.avg("ChainMargin"), 2).alias("AvgWeeklyMargin"),
                F.round(F.avg("ChainTransactions"), 0).alias("AvgWeeklyTransactions"),
                F.round(F.avg("StoreCount"), 0).alias("AvgStoresActive")
            )
            .withColumn("TotalCostPerWeek", F.round(F.col("UnitCost") * F.col("AvgWeeklyUnits"), 2))
            .withColumn("GrossMarginPct", F.round(F.col("AvgWeeklyMargin") / F.col("AvgWeeklySales") * 100, 1))
            .withColumn("MarginPerUnit", F.round(F.col("AvgWeeklyMargin") / F.col("AvgWeeklyUnits"), 4))
            .withColumn("IsLossLeader", F.when(F.col("AvgWeeklyMargin") < 0, 1).otherwise(0))
            .filter(F.col("Product") == "Aussie")
            .withColumn("_gold_ts", F.current_timestamp())
            .orderBy("ActualPrice")
    )
    wg(df, "loss_leader")

    log("gold", rows=0)
    print(f"  ✔ {(datetime.now() - t).seconds}s")


# COMMAND ----------
# MAGIC %md ## 6. Run

# COMMAND ----------

t0 = datetime.now()

print(f"""
╔══════════════════════════════════════════════╗
║  Promotion Pipeline                          ║
║  Run ID : {RUN_ID:<30}║
║  Mode   : {MODE:<30}║
║  Started: {t0.strftime('%Y-%m-%d %H:%M:%S'):<30}║
╚══════════════════════════════════════════════╝
""")

try:
    run_bronze()
except Exception as e:
    log("bronze", status="FAILED", error=str(e))
    raise

try:
    run_silver()
except Exception as e:
    log("silver", status="FAILED", error=str(e))
    raise

try:
    run_gold()
except Exception as e:
    log("gold", status="FAILED", error=str(e))
    raise

print(f"\n✅ Done in {(datetime.now() - t0).seconds}s")


# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT
# MAGIC   run_id,
# MAGIC   run_mode,
# MAGIC   stage,
# MAGIC   rows_out,
# MAGIC   status,
# MAGIC   CAST(started_at AS STRING) AS started_at
# MAGIC FROM workspace.promotion.pipeline_run_log
# MAGIC ORDER BY started_at DESC
# MAGIC LIMIT 10

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT
# MAGIC   table_name,
# MAGIC   last_year,
# MAGIC   last_week,
# MAGIC   rows_processed,
# MAGIC   CAST(updated_at AS STRING) AS updated_at
# MAGIC FROM workspace.promotion.pipeline_watermarks
# MAGIC ORDER BY updated_at DESC